Ao final desta aula, você será capaz de:

Criar DataFrames a partir de listas, dicionários e arquivos CSV.
Entender a diferença entre índice padrão e índice customizado.
Usar set_index e reset_index para controlar o índice.
Renomear colunas com rename.
Verificar e converter tipos de dados com dtypes, astype e to_datetime.
Entender a diferença entre cópia e “view” e por que aparece o famoso SettingWithCopyWarning.



Um DataFrame é uma tabela de dados em memória:

Linhas → registros (ex.: cada venda, cada cliente).
Colunas → campos (ex.: data, valor, cidade).
Índice → “etiqueta” de cada linha (por padrão: 0, 1, 2, …).
Você pode imaginar como uma planilha do Excel, mas com superpoderes de Python.



### Criar dataframe de um dicionario 

In [17]:
import pandas as pd

dados = {
    "id_venda": [1, 2, 3],
    "loja": ["Centro", "Shopping", "Centro"],
    "produto": ["Teclado", "Mouse", "Monitor"],
    "valor": [200.0, 150.0, 900.0]
}

df_vendas = pd.DataFrame(dados)

### Indice padrao 

In [18]:
print(df_vendas.index)

RangeIndex(start=0, stop=3, step=1)


### Índice customizado com set_index

Em muitos casos, faz sentido usar uma coluna como índice, por exemplo id_venda.

In [19]:
df_indexado = df_vendas.set_index("id_venda")
print(df_indexado)

              loja  produto  valor
id_venda                          
1           Centro  Teclado  200.0
2         Shopping    Mouse  150.0
3           Centro  Monitor  900.0


Por padrão, set_index devolve um novo DataFrame — ele não altera o original a menos que você passe inplace=True (que eu costumo evitar com iniciantes, pra ficar mais claro o fluxo).

### Voltando ao índice padrão com reset_index

In [20]:
df_voltou = df_indexado.reset_index()
print(df_voltou)

   id_venda      loja  produto  valor
0         1    Centro  Teclado  200.0
1         2  Shopping    Mouse  150.0
2         3    Centro  Monitor  900.0


### Renomeando colunas com rename

Às vezes os nomes das colunas vêm feios do CSV:

VALOR_VENDA
idVenda
dt_ped
Qtd
Você pode deixá-los mais claros com rename.

In [21]:
df_renomeado = df_vendas.rename(columns={"valor": "valor_venda"})
print(df_renomeado)

   id_venda      loja  produto  valor_venda
0         1    Centro  Teclado        200.0
1         2  Shopping    Mouse        150.0
2         3    Centro  Monitor        900.0


Você também pode renomear mais de uma de uma vez:



In [22]:
df_renomeado = df_vendas.rename(columns={
    "loja": "nome_loja",
    "produto": "nome_produto"
})
print(df_renomeado)

   id_venda nome_loja nome_produto  valor
0         1    Centro      Teclado  200.0
1         2  Shopping        Mouse  150.0
2         3    Centro      Monitor  900.0


Boa prática:
Use nomes de colunas consistentes, tudo minúsculo, sem acento e com _ em vez de espaço.
Ex.: valor_total, data_pedido, cidade_cliente.

# Convertendo tipos com astype

Imagine que id_venda veio como texto (string). Você pode converter:

In [23]:
df_vendas["id_venda"] = df_vendas["id_venda"].astype("int64")

Ou transformar uma coluna em categórica (útil para colunas com poucos valores repetidos, como loja, categoria):

In [24]:
df_vendas["loja"] = df_vendas["loja"].astype("category")
print(df_vendas.dtypes)

id_venda       int64
loja        category
produto       object
valor        float64
dtype: object


# Convertendo datas com to_datetime
Em dados reais, datas quase sempre vêm como texto:

In [25]:
dados = {
    "id_venda": [1, 2, 3],
    "data": ["2024-10-01", "2024-10-02", "2024-10-03"],
    "valor": [200.0, 150.0, 900.0]
}
df_datas = pd.DataFrame(dados)
print(df_datas)
print(df_datas.dtypes)

   id_venda        data  valor
0         1  2024-10-01  200.0
1         2  2024-10-02  150.0
2         3  2024-10-03  900.0
id_venda      int64
data         object
valor       float64
dtype: object


Convertendo:

In [26]:
df_datas["data"] = pd.to_datetime(df_datas["data"])
print(df_datas.dtypes)

id_venda             int64
data        datetime64[ns]
valor              float64
dtype: object


Agora data vira datetime64[ns], e você pode:

Extrair ano, mês, dia.
Ordenar por data.
Fazer filtros de intervalo de tempo.

# Cópia vs “view” e o SettingWithCopyWarning

Essa parte é onde muita gente novata sofre, então vamos com calma.

# O problema clássico

Quando você filtra um DataFrame, às vezes o pandas não tem certeza se você está mexendo na visão (view) do original ou em uma cópia independente.

In [27]:
df_caros = df_vendas[df_vendas["valor"] > 300]
df_caros["valor_com_desconto"] = df_caros["valor"] * 0.9

C:\Users\lucas\AppData\Local\Temp\ipykernel_11696\1379690483.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_caros["valor_com_desconto"] = df_caros["valor"] * 0.9


Isso pode gerar um aviso:

SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

Tradução:

“Opa, você está alterando algo que pode ser só uma visão, não o DataFrame original. Cuidado.”

Não significa que deu tudo errado, mas o pandas está dizendo:

“Talvez você ache que está mudando o original, mas não está.”

# Como evitar o problema (abordagem recomendada

Existem duas boas práticas:
# Usar .loc no DataFrame original

In [28]:
df_vendas.loc[df_vendas["valor"] > 300, "valor_com_desconto"] = df_vendas["valor"] * 0.9

Aqui deixamos claro:

Vamos alterar a coluna valor_com_desconto
Somente nas linhas onde valor > 300
No DataFrame original df_vendas.

# Criar uma cópia explícita
Se você realmente quer trabalhar em um DataFrame separado:

Ao usar .copy(), você diz explicitamente:

“Sim, eu quero um DataFrame independente.”

In [29]:
df_caros = df_vendas[df_vendas["valor"] > 300].copy()
df_caros["valor_com_desconto"] = df_caros["valor"] * 0.9

 # Resumão da aula
Hoje você viu:

Como criar DataFrames a partir de:
dicionários,
listas de dicionários,
CSV.
O que é o índice e como:
usar índice padrão,
definir um índice customizado com set_index,
voltar ao padrão com reset_index.
Como renomear colunas com rename.
Como verificar e converter tipos (dtypes, astype, to_datetime).
A diferença entre cópia e view, e como se proteger do SettingWithCopyWarning usando .loc e .copy().
Esses conceitos são a base para tudo que vem depois: limpeza, análise, visualização, modelos.